# Nicheformer Benchmarking

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 04.12.2024
- Date of Last Modification: 20.02.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

- Run this notebook in the nicheformer conda environment, installable from ```../../envs/nicheformer.yaml```.
- Clone Nicheformer from GitHub into ```<root>/analysis/benchmarking```: ```git clone https://github.com/theislab/nicheformer```.
- Download the pretrained model weights as described here https://github.com/theislab/nicheformer/blob/main/README.md and put them into the ```./nicheformer/data``` directory.
- For zero shot inference, this notebooks follows the notebook from https://github.com/theislab/nicheformer/blob/main/notebooks/experiments/get_embeddings.ipynb

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")
sys.path.append("./nicheformer/src")

In [ ]:
# Execute above cells twice
import os
import warnings
from tqdm import tqdm
from typing import Optional, Dict, Any

import anndata as ad
import numpy as np
import pytorch_lightning as pl
import scanpy as sc
import torch
from torch.utils.data import DataLoader
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

from nicheformer.data import NicheformerDataset
from nicheformer.models import Nicheformer

from analysis_utils import *
from benchmarking_utils import max_overlap_plot

### 1.2 Configure Paths

In [ ]:
st_data_folder_path = '/lustre/scratch126/cellgen/team361/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/lustre/scratch126/cellgen/team361/sb75/nichejepa-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'
nicheformer_folder_path = '/lustre/scratch126/cellgen/team361/sb75/nichejepa-reproducibility/analysis/benchmarking/nicheformer'

BASE_PATH = f'{nicheformer_folder_path}/data/model_means'

### 1.3 Define Config

In [ ]:
config = {
    'data_path': f'{artifacts_folder_path}/nicheformer/chl59-8b_1p.h5ad',
    'technology_mean_path': f"{BASE_PATH}/cosmx_mean_script.npy", #'path/to/technology_mean.npy',  # Path to technology mean file
    'checkpoint_path': f'{nicheformer_folder_path}/data/nicheformer.ckpt',  # Path to model checkpoint
    'output_path': f'{benchmarking_artifacts_folder_path}/nicheformer/chl59-8b_1p_embedded.h5ad',  # Where to save the result, it is a new h5ad
    'output_dir': '.',  # Directory for any intermediate outputs
    'batch_size': 32,
    'max_seq_len': 1500, 
    'aux_tokens': 30, 
    'chunk_size': 1000, # to prevent OOM
    'num_workers': 4,
    'precision': 32,
    'embedding_layer': -1,  # Which layer to extract embeddings from (-1 for last layer)
    'embedding_name': 'embeddings'  # Name suffix for the embedding key in adata.obsm
}

In [ ]:
latent_cluster_key = 'predicted_niche'

### 1.4 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/metrics', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/nicheformer', exist_ok=True)

## 2. Zero-shot Niche Identification

This data is partially in the training data of Nicheformer (no zero-shot).

### 2.1 chl59-8b_1p (NanoString CosMx Human NSCLC)

In [ ]:
# Define dataset params
dataset_name = 'chl59-8b_1p'
batches = ['Lung9_Rep1',
           'Lung9_Rep2',
           'Lung5_Rep1',
           'Lung5_Rep2',
           'Lung6',
           'Lung12']
niche_key = 'niche'
batch_key = 'batch'
spot_sizes = [80 * 0.18] * len(batches)

OUT_PATH = f'{benchmarking_artifacts_folder_path}/nicheformer/{dataset_name}'
os.makedirs(OUT_PATH, exist_ok=True)

In [ ]:
# Define config
config = {
    'data_path': f'{artifacts_folder_path}/nicheformer/{dataset_name}.h5ad',
    'technology_mean_path': f"{BASE_PATH}/cosmx_mean_script.npy", #'path/to/technology_mean.npy',  # Path to technology mean file
    'checkpoint_path': f'{nicheformer_folder_path}/data/nicheformer.ckpt',  # Path to model checkpoint
    'output_path': f'{benchmarking_artifacts_folder_path}/nicheformer/{dataset_name}_embedded.h5ad',  # Where to save the result, it is a new h5ad
    'output_dir': '.',  # Directory for any intermediate outputs
    'batch_size': 32,
    'max_seq_len': 1500, 
    'aux_tokens': 30, 
    'chunk_size': 1000, # to prevent OOM
    'num_workers': 4,
    'precision': 32,
    'embedding_layer': -1,  # Which layer to extract embeddings from (-1 for last layer)
    'embedding_name': 'embeddings'  # Name suffix for the embedding key in adata.obsm
}

#### 2.1.1 Prepare Data

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]

            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata.var_names = adata.var['ensembl_id']
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.18 # conversion rate from https://nanostring-public-share.s3.us-west-2.amazonaws.com/SMI-Compressed/SMI-ReadMe.html

In [ ]:
print(adata.obs[niche_key].value_counts())

#### 2.1.2 Load Data

In [ ]:
model = ad.read_h5ad(f"{BASE_PATH}/model.h5ad")

In [ ]:
# Set random seed for reproducibility
pl.seed_everything(42)

# Load mean for normalization
technology_mean = np.load(config['technology_mean_path'])

# format data properly with the model
adata = ad.concat([model, adata], join='outer', axis=0)
# dropping the first observation 
adata = adata[1:].copy()

mask = adata.var_names.isin(model.var_names)
adata = adata[:, mask].copy()

As a reference, the metadata tokens are

modality_dict = { 'dissociated': 3, 'spatial': 4,}

specie_dict = { 'human': 5, 'Homo sapiens': 5, 'Mus musculus': 6, 'mouse': 6,}

technology_dict = { "merfish": 7, "MERFISH": 7, "cosmx": 8, "NanoString digital spatial profiling": 8, "visium": 9, "10x 5' v2": 10, "10x 3' v3": 11, "10x 3' v2": 12, "10x 5' v1": 13, "10x 3' v1": 14, "10x 3' transcription profiling": 15, "10x transcription profiling": 15, "10x 5' transcription profiling": 16, "CITE-seq": 17, "Smart-seq v4": 18, }

In [ ]:
# Change accordingly
adata.obs['modality'] = 4 # spatial
adata.obs['specie'] = 5 # human
adata.obs['technology'] = 8 # cosmx

In [ ]:
adata.obs['nicheformer_split'] = 'train'

In [ ]:
# Create dataset
dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=1500,
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000)
)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

#### 2.1.3 Retrieve Embeddings

In [ ]:
# Load pre-trained model
model = Nicheformer.load_from_checkpoint(checkpoint_path=config['checkpoint_path'], strict=False)
model.eval()  # Set to evaluation mode

# Configure trainer
trainer = pl.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    default_root_dir=config['output_dir'],
    precision=config.get('precision', 32),
)

In [ ]:
print("Extracting embeddings...")
embeddings = []
device = model.embeddings.weight.device

with torch.no_grad():
    for batch in tqdm(dataloader):
        # Move batch to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}

        # Get embeddings from the model
        emb = model.get_embeddings(
            batch=batch,
            layer=config.get('embedding_layer', -1)  # Default to last layer
        )
        embeddings.append(emb.cpu().numpy())

# Concatenate all embeddings
embeddings = np.concatenate(embeddings, axis=0)

In [ ]:
# Store embeddings in AnnData object
embedding_key = f"X_niche_{config.get('embedding_name', 'embeddings')}"
adata.obsm[embedding_key] = embeddings

In [ ]:
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X_niche_embeddings',
                key_added='X_niche_embeddings')
sc.tl.umap(adata,
           neighbors_key='X_niche_embeddings',
          )

In [ ]:
# Save updated AnnData
adata.write_h5ad(config['output_path'])

print(f"Embeddings saved to {config['output_path']} in obsm['{embedding_key}']")

In [ ]:
adata = sc.read_h5ad(config['output_path'])

#### 2.1.4 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth)
latent_leiden_resolution = 0.5

sc.tl.leiden(adata,
             neighbors_key='X_niche_embeddings',
             key_added=latent_cluster_key,
             resolution=latent_leiden_resolution)
#sc.pl.umap(adata,
#           color='niche')

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/nicheformer/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
inferred_niches_ordered = [
    '2',
    '3',
    '4',
    '0',
    '9',
    '6',
    '8',
    '5',
    '1',
    '11',
    '10',
    '7']

In [ ]:
# Plot nicheformer niches
label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_nicheformer_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.1.5 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_niches_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_niches_list.append(adata.obs[latent_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X_niche_embeddings',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution)

    n_niches_list.append(adata.obs[latent_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_nicheformer_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 2.2 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

In [ ]:
# Define dataset params
dataset_name = 'chr78-11b_1p'
batches = ['CM007',
           'CM008',
           'CM009',
           'CM011',
           'CM013',
           'CM015',
           'CM016',
           'CM017',
           'CM018',
           'CM019',
           'CM020']
niche_key = 'broad_anatomy'
batch_key = 'batch'
spot_sizes = [240 * 0.093] * len(batches)

In [ ]:
# Define config
config = {
    'data_path': f'{artifacts_folder_path}/nicheformer/{dataset_name}.h5ad',
    'technology_mean_path': f"{BASE_PATH}/xenium_mean_script.npy", #'path/to/technology_mean.npy',  # Path to technology mean file
    'checkpoint_path': f'{nicheformer_folder_path}/data/nicheformer.ckpt',  # Path to model checkpoint
    'output_path': f'{benchmarking_artifacts_folder_path}/nicheformer/{dataset_name}_embedded.h5ad',  # Where to save the result, it is a new h5ad
    'output_dir': '.',  # Directory for any intermediate outputs
    'batch_size': 32,
    'max_seq_len': 1500, 
    'aux_tokens': 30, 
    'chunk_size': 1000, # to prevent OOM
    'num_workers': 4,
    'precision': 32,
    'embedding_layer': -1,  # Which layer to extract embeddings from (-1 for last layer)
    'embedding_name': 'embeddings'  # Name suffix for the embedding key in adata.obsm
}

#### 2.2.1 Prepare Data

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata.var_names = adata.var['ensembl_id']
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.093 # conversion rate from Valentina

# Combine regions
adata.obs[niche_key] = adata.obs[niche_key].apply(lambda x: x.replace('Urogenital Sinus', 'UrogenitalSinus'))

In [ ]:
print(adata.obs[niche_key].value_counts())

#### 2.2.2 Load Data

In [ ]:
model = ad.read_h5ad(f"{BASE_PATH}/model.h5ad")

In [ ]:
# Set random seed for reproducibility
pl.seed_everything(42)

# Load mean for normalization
technology_mean = np.load(config['technology_mean_path'])

# format data properly with the model
adata = ad.concat([model, adata], join='outer', axis=0)
# dropping the first observation 
adata = adata[1:].copy()

mask = adata.var_names.isin(model.var_names)
adata = adata[:, mask].copy()

As a reference, the metadata tokens are

modality_dict = { 'dissociated': 3, 'spatial': 4,}

specie_dict = { 'human': 5, 'Homo sapiens': 5, 'Mus musculus': 6, 'mouse': 6,}

technology_dict = { "merfish": 7, "MERFISH": 7, "cosmx": 8, "NanoString digital spatial profiling": 8, "visium": 9, "10x 5' v2": 10, "10x 3' v3": 11, "10x 3' v2": 12, "10x 5' v1": 13, "10x 3' v1": 14, "10x 3' transcription profiling": 15, "10x transcription profiling": 15, "10x 5' transcription profiling": 16, "CITE-seq": 17, "Smart-seq v4": 18, }

In [ ]:
# Change accordingly
# Pick Xenium as ISS Cartana is not available
adata.obs['modality'] = 4 # spatial
adata.obs['specie'] = 5 # human
adata.obs['technology'] = 9 # xenium (as per https://github.com/theislab/nicheformer/blob/main/notebooks/tokenization/xenium_human_lung.ipynb)

In [ ]:
adata.obs['nicheformer_split'] = 'train'

In [ ]:
# Create dataset
dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=1500,
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000)
)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

#### 2.2.3 Retrieve Embeddings

In [ ]:
# Load pre-trained model
model = Nicheformer.load_from_checkpoint(checkpoint_path=config['checkpoint_path'], strict=False)
model.eval()  # Set to evaluation mode

# Configure trainer
trainer = pl.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    default_root_dir=config['output_dir'],
    precision=config.get('precision', 32),
)

In [ ]:
print("Extracting embeddings...")
embeddings = []
device = model.embeddings.weight.device

with torch.no_grad():
    for batch in tqdm(dataloader):
        # Move batch to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}

        # Get embeddings from the model
        emb = model.get_embeddings(
            batch=batch,
            layer=config.get('embedding_layer', -1)  # Default to last layer
        )
        embeddings.append(emb.cpu().numpy())


# Concatenate all embeddings
embeddings = np.concatenate(embeddings, axis=0)

In [ ]:
# Store embeddings in AnnData object
embedding_key = f"X_niche_{config.get('embedding_name', 'embeddings')}"
adata.obsm[embedding_key] = embeddings

# Save updated AnnData
adata.write_h5ad(config['output_path'])

print(f"Embeddings saved to {config['output_path']} in obsm['{embedding_key}']")

In [ ]:
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X_niche_embeddings',
                key_added='X_niche_embeddings')
sc.tl.umap(adata,
           neighbors_key='X_niche_embeddings',
          )

In [ ]:
sc.pl.umap(adata,
           color='niche')

Embeddings collapse with Xenium mean. Stop here.

#### 2.2.4 Highlight Run (Matching Ground Truth Number of Niches)

#### 2.2.5 Quantitative Performance

### 2.3 xhs49-1b_1p (Xenium Human Melanoma)

In [ ]:
dataset_name = 'xhs49-1b_1p'
niche_key = 'niche_name'
spot_size = 20

In [ ]:
# Define config
config = {
    'data_path': f'{artifacts_folder_path}/nicheformer/{dataset_name}.h5ad',
    'technology_mean_path': f"{BASE_PATH}/xenium_mean_script.npy", #'path/to/technology_mean.npy',  # Path to technology mean file
    'checkpoint_path': f'{nicheformer_folder_path}/data/nicheformer.ckpt',  # Path to model checkpoint
    'output_path': f'{benchmarking_artifacts_folder_path}/nicheformer/{dataset_name}_embedded.h5ad',  # Where to save the result, it is a new h5ad
    'output_dir': '.',  # Directory for any intermediate outputs
    'batch_size': 32,
    'max_seq_len': 1500, 
    'aux_tokens': 30, 
    'chunk_size': 1000, # to prevent OOM
    'num_workers': 4,
    'precision': 32,
    'embedding_layer': -1,  # Which layer to extract embeddings from (-1 for last layer)
    'embedding_name': 'embeddings'  # Name suffix for the embedding key in adata.obsm
}

#### 2.3.1 Prepare Data

In [ ]:
adata = sc.read_h5ad(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/{dataset_name}/Xenium_Prime_Human_Skin_FFPE_outs.h5ad')

# Filter cells without niche labels
adata = adata[~adata.obs['niche_name'].isna()]

adata.var_names = adata.var['ensembl_id']

In [ ]:
print(adata.obs[niche_key].value_counts())

#### 2.3.2 Load Data

In [ ]:
model = ad.read_h5ad(f"{BASE_PATH}/model.h5ad")

In [ ]:
# Set random seed for reproducibility
pl.seed_everything(42)

# Load mean for normalization
technology_mean = np.load(config['technology_mean_path'])

# format data properly with the model
adata = ad.concat([model, adata], join='outer', axis=0)
# dropping the first observation 
adata = adata[1:].copy()

mask = adata.var_names.isin(model.var_names)
adata = adata[:, mask].copy()

As a reference, the metadata tokens are

modality_dict = { 'dissociated': 3, 'spatial': 4,}

specie_dict = { 'human': 5, 'Homo sapiens': 5, 'Mus musculus': 6, 'mouse': 6,}

technology_dict = { "merfish": 7, "MERFISH": 7, "cosmx": 8, "NanoString digital spatial profiling": 8, "visium": 9, "10x 5' v2": 10, "10x 3' v3": 11, "10x 3' v2": 12, "10x 5' v1": 13, "10x 3' v1": 14, "10x 3' transcription profiling": 15, "10x transcription profiling": 15, "10x 5' transcription profiling": 16, "CITE-seq": 17, "Smart-seq v4": 18, }

In [ ]:
# Change accordingly
adata.obs['modality'] = 4 # spatial
adata.obs['specie'] = 5 # human
adata.obs['technology'] = 9 # xenium (as per https://github.com/theislab/nicheformer/blob/main/notebooks/tokenization/xenium_human_lung.ipynb)

In [ ]:
adata.obs['nicheformer_split'] = 'train'

In [ ]:
# Create dataset
dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=1500,
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000)
)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

#### 2.3.3 Retrieve Embeddings

In [ ]:
# Load pre-trained model
model = Nicheformer.load_from_checkpoint(checkpoint_path=config['checkpoint_path'], strict=False)
model.eval()  # Set to evaluation mode

# Configure trainer
trainer = pl.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    default_root_dir=config['output_dir'],
    precision=config.get('precision', 32),
)

In [ ]:
print("Extracting embeddings...")
embeddings = []
device = model.embeddings.weight.device

with torch.no_grad():
    for batch in tqdm(dataloader):
        # Move batch to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}

        # Get embeddings from the model
        emb = model.get_embeddings(
            batch=batch,
            layer=config.get('embedding_layer', -1)  # Default to last layer
        )
        embeddings.append(emb.cpu().numpy())


# Concatenate all embeddings
embeddings = np.concatenate(embeddings, axis=0)

In [ ]:
# Store embeddings in AnnData object
embedding_key = f"X_niche_{config.get('embedding_name', 'embeddings')}"
adata.obsm[embedding_key] = embeddings

In [ ]:
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X_niche_embeddings',
                key_added='X_niche_embeddings')
sc.tl.umap(adata,
           neighbors_key='X_niche_embeddings',
          )

In [ ]:
# Save updated AnnData
adata.write_h5ad(config['output_path'])

print(f"Embeddings saved to {config['output_path']} in obsm['{embedding_key}']")

#### 2.3.4 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth - 7)
latent_leiden_resolution = 0.1

sc.tl.leiden(adata,
             neighbors_key='X_niche_embeddings',
             key_added=latent_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/nicheformer/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cluster_key].unique().tolist())
print(filtered_df[latent_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
    '5',
    '3',
    '1',
    '4',
    '0',
    '2',
    '6',]

In [ ]:
# Plot nicheformer niches
label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

file_name = f'{dataset_name}_nicheformer_niches.svg'
sc.pl.spatial(adata,
              color=latent_cluster_key,
              palette=pred_palette,
              spot_size=spot_size,
              save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.3.5 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_niches_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_niches_list.append(adata.obs[latent_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X_niche_embeddings',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution)

    n_niches_list.append(adata.obs[latent_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_nicheformer_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 2.4 xhs1000-39b_1p (Xenium Human Skin - Eczema)

In [ ]:
# Define dataset params
dataset_name = 'xhs1000-39b_1p'
batches = ['batch11',
           'batch19',
           'batch32']
niche_key = 'niche_type'
batch_key = 'sample'
spot_sizes = [20] * len(batches)

In [ ]:
# Define config
config = {
    'data_path': f'{artifacts_folder_path}/nicheformer/{dataset_name}.h5ad',
    'technology_mean_path': f"{BASE_PATH}/xenium_mean_script.npy", #'path/to/technology_mean.npy',  # Path to technology mean file
    'checkpoint_path': f'{nicheformer_folder_path}/data/nicheformer.ckpt',  # Path to model checkpoint
    'output_path': f'{benchmarking_artifacts_folder_path}/nicheformer/{dataset_name}_embedded.h5ad',  # Where to save the result, it is a new h5ad
    'output_dir': '.',  # Directory for any intermediate outputs
    'batch_size': 32,
    'max_seq_len': 1500, 
    'aux_tokens': 30, 
    'chunk_size': 1000, # to prevent OOM
    'num_workers': 4,
    'precision': 32,
    'embedding_layer': -1,  # Which layer to extract embeddings from (-1 for last layer)
    'embedding_name': 'embeddings'  # Name suffix for the embedding key in adata.obsm
}

#### 2.4.1 Prepare Data

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata.var_names = adata.var['ensembl_id']
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial']

In [ ]:
print(adata.obs[niche_key].value_counts())

#### 2.4.2 Load Data

In [ ]:
model = ad.read_h5ad(f"{BASE_PATH}/model.h5ad")

In [ ]:
# Set random seed for reproducibility
pl.seed_everything(42)

# Load mean for normalization
technology_mean = np.load(config['technology_mean_path'])

# format data properly with the model
adata = ad.concat([model, adata], join='outer', axis=0)
# dropping the first observation 
adata = adata[1:].copy()

mask = adata.var_names.isin(model.var_names)
adata = adata[:, mask].copy()

As a reference, the metadata tokens are

modality_dict = { 'dissociated': 3, 'spatial': 4,}

specie_dict = { 'human': 5, 'Homo sapiens': 5, 'Mus musculus': 6, 'mouse': 6,}

technology_dict = { "merfish": 7, "MERFISH": 7, "cosmx": 8, "NanoString digital spatial profiling": 8, "visium": 9, "10x 5' v2": 10, "10x 3' v3": 11, "10x 3' v2": 12, "10x 5' v1": 13, "10x 3' v1": 14, "10x 3' transcription profiling": 15, "10x transcription profiling": 15, "10x 5' transcription profiling": 16, "CITE-seq": 17, "Smart-seq v4": 18, }

In [ ]:
# Change accordingly
# Pick Xenium as ISS Cartana is not available
adata.obs['modality'] = 4 # spatial
adata.obs['specie'] = 5 # human
adata.obs['technology'] = 9 # xenium (as per https://github.com/theislab/nicheformer/blob/main/notebooks/tokenization/xenium_human_lung.ipynb)

In [ ]:
adata.obs['nicheformer_split'] = 'train'

In [ ]:
# Create dataset
dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=1500,
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000)
)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

#### 2.4.3 Retrieve Embeddings

In [ ]:
# Load pre-trained model
model = Nicheformer.load_from_checkpoint(checkpoint_path=config['checkpoint_path'], strict=False)
model.eval()  # Set to evaluation mode

# Configure trainer
trainer = pl.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    default_root_dir=config['output_dir'],
    precision=config.get('precision', 32),
)

In [ ]:
print("Extracting embeddings...")
embeddings = []
device = model.embeddings.weight.device

with torch.no_grad():
    for batch in tqdm(dataloader):
        # Move batch to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}

        # Get embeddings from the model
        emb = model.get_embeddings(
            batch=batch,
            layer=config.get('embedding_layer', -1)  # Default to last layer
        )
        embeddings.append(emb.cpu().numpy())


# Concatenate all embeddings
embeddings = np.concatenate(embeddings, axis=0)

In [ ]:
# Store embeddings in AnnData object
embedding_key = f"X_niche_{config.get('embedding_name', 'embeddings')}"
adata.obsm[embedding_key] = embeddings

# Save updated AnnData
adata.write_h5ad(config['output_path'])

print(f"Embeddings saved to {config['output_path']} in obsm['{embedding_key}']")

In [ ]:
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X_niche_embeddings',
                key_added='X_niche_embeddings')
sc.tl.umap(adata,
           neighbors_key='X_niche_embeddings',
          )

#### 2.4.4 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth - 11)
latent_leiden_resolution = 0.2

sc.tl.leiden(adata,
             neighbors_key='X_niche_embeddings',
             key_added=latent_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cluster_key].unique().tolist())
print(filtered_df[latent_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
    '1',
    '0',
    '3',
    '4',
    '2',
    '10',
    '6',
    '5',
    '9',
    '8',
    '7',]

In [ ]:
# Plot nicheformer niches
label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_nicheformer_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.4.5 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_niches_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_niches_list.append(adata.obs[latent_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X_niche_embeddings',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution)

    n_niches_list.append(adata.obs[latent_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")